# 🐍 Master Guide: Exception Handling (`try`, `except`, `else`, `finally`)
> **Single-Source Revision Guide**: Covers Python's exception hierarchy, zero-cost exception handling in Python 3.11+, exception chaining, and interview traps.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Exceptions signal runtime anomalies. Robust applications catch errors gracefully using the 4-part block: `try`, `except`, `else`, and `finally`.

---
## 🧠 2. CPython Architecture: The Exception Hierarchy & Zero-Cost Exceptions

### Python's Exception Tree
```
BaseException (Root: KeyboardInterrupt, SystemExit)
 └── Exception (Standard root for application errors)
      ├── ArithmeticError (ZeroDivisionError, OverflowError)
      ├── LookupError (IndexError, KeyError)
      ├── ValueError
      ├── TypeError
      └── OSError (FileNotFoundError, PermissionError)
```

- **Why Catch `Exception` and NOT `BaseException`**: Catching `BaseException` intercepts `KeyboardInterrupt` (`Ctrl+C`) and `SystemExit`, making your process impossible to terminate cleanly!
- **Zero-Cost Exceptions (Python 3.11+)**: Entering a `try` block has **ZERO runtime cost** if no exception occurs! CPython stores exception tables in the code object without runtime setup instructions.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: The Complete 4-Part Structure (`try/except/else/finally`)
- `try`: Code that might raise an exception.
- `except`: Executes ONLY if an error was raised in `try`.
- `else`: Executes ONLY if NO error was raised in `try`.
- `finally`: Executes ALWAYS, regardless of errors or returns!

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"[Except] Handled error: {e}")
        return None
    else:
        print(f"[Else] Success! Result is {result}")
        return result
    finally:
        print("[Finally] Cleanup always executed!")

print("--- Call 1 (Success) ---")
safe_divide(10, 2)

print("\n--- Call 2 (Error) ---")
safe_divide(10, 0)

---
## 🟡 4. Part 2: Intermediate Features

### Exception Chaining (`raise ... from ...`)
Python preserves the original root cause traceback when raising a higher-level exception.

In [ ]:
def fetch_database_record(record_id):
    try:
        # Simulate lower-level error:
        int(record_id)
    except ValueError as original_error:
        # Chain to a higher-level domain exception:
        raise RuntimeError(f"Failed to fetch record '{record_id}'") from original_error

try:
    fetch_database_record("abc_not_an_int")
except RuntimeError as e:
    print("Caught chained exception:", e)
    print("Original cause was     :", repr(e.__cause__))

### Suppressing Exception Context (`raise ... from None`)
Hides confusing internal tracebacks from end users.

In [ ]:
def parse_config(cfg):
    try:
        return cfg["port"]
    except KeyError:
        # Suppresses the original KeyError traceback:
        raise ValueError("Invalid configuration: 'port' missing!") from None

try:
    parse_config({})
except ValueError as e:
    print("Caught clean exception (KeyError suppressed):", e)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Bare `except:` Anti-Pattern
A bare `except:` or `except BaseException:` catches `KeyboardInterrupt` and `SystemExit`, making your program impossible to stop with `Ctrl+C`!

In [ ]:
print("Rule: Never write 'except:' or 'except BaseException:'. Always use 'except Exception:'!")

### ⚠️ Gotcha 2: Exception Subclass Order Trap
Python checks `except` blocks from top to bottom. If a parent class (like `LookupError`) is placed before a child class (like `KeyError`), the child block will **never execute**!

In [ ]:
d = {}
try:
    val = d["missing"]
except LookupError:
    print("Caught by LookupError because LookupError is the parent of KeyError!")
except KeyError:
    print("This line will NEVER be reached!")

print("Rule: Always catch specific child exceptions BEFORE general parent exceptions!")

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| try...except...else...finally      | else = no error; finally = always runs|
| except Exception:                  | Correct root for app errors           |
| Bare except:                       | DANGEROUS! Swallows Ctrl+C and exit   |
| raise NewExc from Original         | Preserves root cause in __cause__     |
| raise NewExc from None             | Suppresses internal traceback noise   |
| Exception ordering                 | Catch specific before general parents |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. The 6 Forms of Exception Handling (`DivEx1.py` - `DivEx6.py`)
> **The 6 Exception Architectures**:
> 1. `try` with single specific `except` block.
> 2. `try` with multiple independent `except` blocks.
> 3. `try` with aliased exception object (`except ValueError as err`).
> 4. `try` with tuple of multiple exceptions (`except (ZeroDivisionError, ValueError):`).
> 5. `try - except - else` (Executes `else` only when NO exception occurs).
> 6. `try - except - else - finally` (`finally` ALWAYS executes for cleanup).

In [ ]:
# The 6 Exception Handling Forms Drill (DivEx1 - DivEx6)
def safe_division_tester(val_a, val_b):
    print(f"\nTesting division: safe_division_tester({val_a!r}, {val_b!r})")
    try:
        a = float(val_a)
        b = float(val_b)
        res = a / b
    except (ZeroDivisionError, ValueError) as err:
        print(f"  [EXCEPT] Caught Known Error: {type(err).__name__} -> {err}")
    except Exception as e:
        print(f"  [EXCEPT] Caught Unexpected General Error: {e}")
    else:
        print(f"  [ELSE] Calculation Succeeded: {a} / {b} = {res:.4f}")
    finally:
        print("  [FINALLY] Resource cleanup complete.")

safe_division_tester("100", "20")    # Valid -> executes else & finally
safe_division_tester("100", "0")     # ZeroDivisionError -> executes except & finally
safe_division_tester("abc", "10")    # ValueError -> executes except & finally